# LeetCode 253 — Meeting Rooms II bằng Python

Link đề: https://leetcode.com/problems/meeting-rooms-ii/ · Độ khó: **Medium** (Premium) · Chủ đề: Array, Sorting, Heap (Priority Queue), Sweep Line, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minMeetingRooms(self, intervals: List[List[int]]) -> int:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (đếm số cuộc họp đang diễn ra tại **mọi** mốc thời gian nguyên).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho các cuộc họp `[start, end]`, tìm **số phòng họp ít nhất** để mọi cuộc họp diễn ra; hai cuộc họp **chồng lấn** thì không thể dùng chung một phòng, còn cuộc họp kết thúc đúng lúc cuộc khác bắt đầu **thì không chồng lấn**.
2. Nhận ra đáp án chính là **số cuộc họp diễn ra đồng thời nhiều nhất** (độ sâu chồng lấn lớn nhất, maximum concurrency).
3. Đi từ cách **đếm chồng lấn tại từng điểm bắt đầu** `O(n²)` → **sắp xếp + min-heap các giờ kết thúc** `O(n log n)` → **hai con trỏ trên hai mảng đã sắp (start, end)** → **sự kiện quét (sweep line)** → **mảng hiệu (difference array)** theo miền thời gian.
4. Làm quen với `heapq`, sắp xếp theo khóa, quy ước **khoảng nửa mở `[start, end)`** và cách xử lý **hòa (tie)** khi `end == start`.
5. Tránh các lỗi kinh điển: dùng `<` thay vì `<=` khi so với giờ kết thúc nhỏ nhất, quên sắp xếp, đếm **số cặp chồng lấn** thay vì số chồng lấn đồng thời, xếp sự kiện kết thúc sau sự kiện bắt đầu khi hòa.

## 2. Đề bài & ví dụ

Cho mảng các khoảng thời gian họp `intervals`, trong đó `intervals[i] = [start_i, end_i]`, trả về **số phòng họp tối thiểu** cần dùng.

**Ví dụ 1**

- Input: `intervals = [[0,30],[5,10],[15,20]]`
- Output: `2`
- Giải thích: cuộc họp `[0,30]` chiếm một phòng suốt cả khoảng; hai cuộc họp `[5,10]` và `[15,20]` không chồng nhau nên dùng chung phòng thứ hai.

**Ví dụ 2**

- Input: `intervals = [[7,10],[2,4]]`
- Output: `1`
- Giải thích: hai cuộc họp không chồng nhau nên một phòng là đủ.

**Ràng buộc:** `1 <= intervals.length <= 10^4`; `0 <= start_i < end_i <= 10^6`.

**Quy ước quan trọng:** khoảng thời gian là **nửa mở** `[start, end)`: cuộc họp `[1, 5]` và `[5, 8]` **không** chồng lấn, nên dùng chung một phòng.

In [1]:
from typing import List
import heapq
import random
import time

examples = [
    ([[0, 30], [5, 10], [15, 20]], 2),
    ([[7, 10], [2, 4]], 1),
]
for intervals, expected in examples:
    print(f"intervals={intervals} -> kỳ vọng {expected}")

intervals=[[0, 30], [5, 10], [15, 20]] -> kỳ vọng 2
intervals=[[7, 10], [2, 4]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Khoảng thời gian nửa mở `[start, end)` và điều kiện chồng lấn

Hai khoảng `[s1, e1)` và `[s2, e2)` **chồng lấn** khi và chỉ khi `s1 < e2` **và** `s2 < e1`. Nếu `e1 == s2` thì `s2 < e1` **sai** nên không chồng lấn: cuộc họp 1 vừa kết thúc thì cuộc họp 2 bắt đầu, dùng chung phòng được.
#### Code demo

In [2]:
def overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]


print(overlap([0, 30], [5, 10]))     # True: [5, 10] nằm trong [0, 30]
print(overlap([1, 5], [5, 8]))       # False: chạm đầu mút nhưng không chồng
print(overlap([1, 5], [4, 8]))       # True
print(overlap([7, 10], [2, 4]))      # False

True
False
True
False


### 3.2. Sắp xếp theo khóa: `sorted(..., key=...)`

`sorted(intervals, key=lambda x: x[0])` sắp các cuộc họp theo **giờ bắt đầu** mà không đổi danh sách gốc (`list.sort` thì sắp **tại chỗ**). Sắp xếp thường là bước đầu của bài khoảng thời gian: sau khi sắp theo giờ bắt đầu ta xét các cuộc họp "theo dòng thời gian".
#### Code demo

In [3]:
intervals = [[15, 20], [0, 30], [5, 10]]
print(sorted(intervals, key=lambda x: x[0]))      # [[0, 30], [5, 10], [15, 20]]
print(intervals)                                    # không đổi
print(sorted(s for s, e in intervals), sorted(e for s, e in intervals))   # [0, 5, 15] [10, 20, 30]

[[0, 30], [5, 10], [15, 20]]
[[15, 20], [0, 30], [5, 10]]
[0, 5, 15] [10, 20, 30]


### 3.3. Min-heap `heapq`

**Heap nhỏ nhất (min-heap)** cho phép thêm phần tử và lấy **phần tử nhỏ nhất** trong `O(log k)`: `heappush(h, x)`, `heappop(h)`, `h[0]` xem nhỏ nhất, `heapreplace(h, x)` (pop rồi push `x` trong một bước). Ở bài này heap sẽ chứa **giờ kết thúc của các cuộc họp đang chiếm phòng**; `h[0]` là phòng **sớm rảnh nhất**.
#### Code demo

In [4]:
h = []
for x in [30, 10, 20]:
    heapq.heappush(h, x)
print(h[0], len(h))              # 10 3
print(heapq.heappop(h), h[0])    # 10 20
heapq.heapreplace(h, 25)         # bỏ 20 (nhỏ nhất), thêm 25
print(sorted(h))                 # [25, 30]

10 3
10 20
[25, 30]


### 3.3b. Độ sâu chồng lấn lớn nhất = số phòng cần

Mỗi cuộc họp chiếm một phòng suốt khoảng của nó. Tại một thời điểm `t`, nếu có `k` cuộc họp đang diễn ra thì cần **ít nhất `k` phòng**. Vậy số phòng `>=` max theo `t` của `k`. Ngược lại, có thể gán phòng sao cho đúng bằng giá trị đó (chứng minh ở mục 10). Nên: **đáp án = số cuộc họp đồng thời nhiều nhất**.
#### Code demo

In [5]:
meetings = [[0, 30], [5, 10], [15, 20]]
for t in (0, 5, 9, 10, 15, 19, 20, 25, 29):
    running = [m for m in meetings if m[0] <= t < m[1]]
    print(f"t={t:>2}: {len(running)} cuộc họp đang diễn ra {running}")

t= 0: 1 cuộc họp đang diễn ra [[0, 30]]
t= 5: 2 cuộc họp đang diễn ra [[0, 30], [5, 10]]
t= 9: 2 cuộc họp đang diễn ra [[0, 30], [5, 10]]
t=10: 1 cuộc họp đang diễn ra [[0, 30]]
t=15: 2 cuộc họp đang diễn ra [[0, 30], [15, 20]]
t=19: 2 cuộc họp đang diễn ra [[0, 30], [15, 20]]
t=20: 1 cuộc họp đang diễn ra [[0, 30]]
t=25: 1 cuộc họp đang diễn ra [[0, 30]]
t=29: 1 cuộc họp đang diễn ra [[0, 30]]


### 3.4. Sự kiện (event) và sắp xếp bộ `(thời_gian, loại)`

Có thể biến mỗi cuộc họp thành **hai sự kiện**: `(start, +1)` và `(end, -1)`. Sắp theo thời gian rồi cộng dồn `+1/-1`, giá trị lớn nhất của tổng cộng dồn là độ sâu chồng lấn. Sắp xếp bộ `(t, d)` mặc định so `t` trước rồi `d`; vì `-1 < +1` nên **khi cùng thời điểm, sự kiện kết thúc (`-1`) đứng trước sự kiện bắt đầu (`+1`)** — đúng quy ước nửa mở.
#### Code demo

In [6]:
events = []
for s, e in [[1, 5], [5, 8]]:
    events += [(s, 1), (e, -1)]
print(sorted(events))            # [(1, 1), (5, -1), (5, 1), (8, -1)]: tại 5 kết thúc trước, bắt đầu sau
cur = best = 0
for t, d in sorted(events):
    cur += d
    best = max(best, cur)
print(best)                      # 1

[(1, 1), (5, -1), (5, 1), (8, -1)]
1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `[[0,30],[5,10],[15,20]]`. Vẽ trục thời gian:

```
0                                30
[================================)     phòng A (cuộc họp 1)
     5     10
     [=====)                            cuộc họp 2
                15     20
                [=====)                 cuộc họp 3
```

- Tại `t = 7`: đang diễn ra `[0,30]` và `[5,10]` ⇒ `2` cuộc họp.
- Tại `t = 17`: `[0,30]` và `[15,20]` ⇒ `2` cuộc họp.
- Không có thời điểm nào có `3` cuộc họp (vì `[5,10]` và `[15,20]` không giao nhau).

Vậy cần **2 phòng**: phòng A giữ `[0,30]`; phòng B giữ `[5,10]` rồi tái sử dụng cho `[15,20]` (vì `10 <= 15`).

**Cách nghĩ then chốt:** đi dọc thời gian; mỗi khi một cuộc họp **bắt đầu**, kiểm tra xem có phòng nào đã **rảnh** (cuộc họp trong phòng đó đã kết thúc) không; nếu có thì dùng lại, nếu không thì mở phòng mới. Phòng nào **rảnh sớm nhất** thì xét đầu tiên: đó là chỗ cần cấu trúc **min-heap**.

## 5. Cách 1 — Brute Force / baseline: đếm chồng lấn tại điểm bắt đầu của từng cuộc họp

**Ý tưởng.** Độ sâu chồng lấn lớn nhất luôn đạt **tại một giờ bắt đầu** nào đó (khi có cuộc họp mới bắt đầu thì số họp đang chạy mới có thể tăng). Với mỗi cuộc họp `i`, đếm số cuộc họp `j` thỏa `start_j <= start_i < end_j`; đáp án là giá trị lớn nhất.
#### Code demo

In [7]:
def rooms_bruteforce(intervals):
    best = 0
    for s, _ in intervals:
        running = 0
        for s2, e2 in intervals:
            if s2 <= s < e2:                    # cuộc họp 2 đang diễn ra tại thời điểm s
                running += 1
        best = max(best, running)
    return best


for intervals, expected in examples:
    print(intervals, "->", rooms_bruteforce(intervals), "(kỳ vọng", expected, ")")

[[0, 30], [5, 10], [15, 20]] -> 2 (kỳ vọng 2 )
[[7, 10], [2, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Gọi `n = len(intervals)`: hai vòng lồng nhau ⇒ **Time `O(n²)`**; với `n = 10^4` là `10^8` phép — rất chậm trong Python; **Space `O(1)`**.

**Nút thắt.** Mỗi lần xét một thời điểm ta **quét lại cả danh sách**. Nếu đã sắp xếp theo thời gian, ta chỉ cần **theo dõi dần** số phòng đang bận.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp theo giờ bắt đầu, rồi duy trì tập các phòng đang bận** (mỗi phòng nhớ **giờ kết thúc** của cuộc họp hiện tại):

- Với cuộc họp mới `[s, e]`: lấy phòng **rảnh sớm nhất** `h[0]` (giờ kết thúc nhỏ nhất).
  - Nếu `h[0] <= s`: phòng đó đã rảnh → **dùng lại**, cập nhật giờ kết thúc thành `e`.
  - Nếu `h[0] > s`: mọi phòng đều đang bận → **mở phòng mới** (push `e`).
- Đáp án là **số phần tử cuối cùng của heap** (số phòng đã mở).

Nhận xét thêm: ta **không cần biết cuộc họp nào ở phòng nào**; chỉ cần đếm. Điều này dẫn tới hai cách "không heap": **hai mảng đã sắp (start, end)** và **sự kiện quét**; và nếu miền thời gian nhỏ (`<= 10^6`) thì **mảng hiệu**.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + min-heap giờ kết thúc

**Giải thích.** Như mục 6. `heapq.heapreplace(h, e)` làm "bỏ phòng rảnh sớm nhất, nhận cuộc họp mới" trong một thao tác.
#### Code demo

In [8]:
def rooms_heap(intervals):
    heap = []                                       # giờ kết thúc của các phòng đang dùng
    for s, e in sorted(intervals, key=lambda x: x[0]):
        if heap and heap[0] <= s:                   # phòng rảnh sớm nhất đã rảnh (kết thúc <= giờ bắt đầu)
            heapq.heapreplace(heap, e)              # dùng lại phòng đó
        else:
            heapq.heappush(heap, e)                 # mở phòng mới
    return len(heap)


for intervals, expected in examples:
    print(intervals, "->", rooms_heap(intervals), "(kỳ vọng", expected, ")")

[[0, 30], [5, 10], [15, 20]] -> 2 (kỳ vọng 2 )
[[7, 10], [2, 4]] -> 1 (kỳ vọng 1 )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def minMeetingRooms(self, intervals: List[List[int]]) -> int:
        heap = []
        for s, e in sorted(intervals, key=lambda x: x[0]):
            if heap and heap[0] <= s:
                heapq.heapreplace(heap, e)
            else:
                heapq.heappush(heap, e)
        return len(heap)


print(Solution().minMeetingRooms([[0, 30], [5, 10], [15, 20]]))   # 2

2


**Độ phức tạp Cách 2:** sắp xếp `O(n log n)`; mỗi cuộc họp một thao tác heap `O(log n)` ⇒ **Time `O(n log n)`**, **Space `O(n)`** (heap có thể chứa tới `n` phần tử).

### Cách 3 — Hai mảng đã sắp (giờ bắt đầu, giờ kết thúc) và hai con trỏ

**Giải thích.** Sắp **riêng** mảng `starts` và mảng `ends` (không cần giữ cặp!). Duyệt các `start` tăng dần; con trỏ `j` chỉ vào giờ kết thúc nhỏ nhất **chưa được "giải phóng"**. Nếu `start < ends[j]` thì chưa có phòng nào rảnh ⇒ cần thêm phòng; ngược lại phòng của `ends[j]` rảnh, dùng lại và dịch `j`.
#### Code demo

In [10]:
def rooms_two_pointers(intervals):
    starts = sorted(s for s, _ in intervals)
    ends = sorted(e for _, e in intervals)
    rooms = 0
    j = 0                                             # chỉ vào giờ kết thúc nhỏ nhất chưa dùng lại
    for s in starts:
        if s < ends[j]:
            rooms += 1                                # chưa phòng nào rảnh -> mở thêm
        else:
            j += 1                                    # một phòng đã rảnh -> dùng lại (không mở thêm)
    return rooms


for intervals, expected in examples:
    print(intervals, "->", rooms_two_pointers(intervals), "(kỳ vọng", expected, ")")

[[0, 30], [5, 10], [15, 20]] -> 2 (kỳ vọng 2 )
[[7, 10], [2, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(n log n)`** (sắp xếp hai mảng; hai con trỏ `O(n)`), **Space `O(n)`**. Không cần heap.

### Cách 4 — Sự kiện quét (sweep line)

**Giải thích.** Mỗi cuộc họp tạo `(start, +1)` và `(end, -1)`; sắp xếp; cộng dồn; lấy giá trị lớn nhất. Khi hòa thời gian, `-1` đứng trước `+1` nhờ thứ tự mặc định của bộ.
#### Code demo

In [11]:
def rooms_sweep(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e, -1))
    events.sort()                                     # cùng thời điểm: -1 (kết thúc) trước +1 (bắt đầu)
    current = best = 0
    for _, delta in events:
        current += delta
        best = max(best, current)
    return best


for intervals, expected in examples:
    print(intervals, "->", rooms_sweep(intervals), "(kỳ vọng", expected, ")")

[[0, 30], [5, 10], [15, 20]] -> 2 (kỳ vọng 2 )
[[7, 10], [2, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n log n)`**, **Space `O(n)`** (`2n` sự kiện).

### Cách 5 — Mảng hiệu (difference array) theo miền thời gian

**Giải thích.** Vì mọi mốc thời gian là số nguyên trong `[0, 10^6]`, dựng mảng `diff` kích thước `max_end + 1`: với mỗi cuộc họp `diff[s] += 1`, `diff[e] -= 1`. Tổng tiền tố `diff[0] + ... + diff[t]` là số cuộc họp đang diễn ra tại `t` (nửa mở nên `e` đã trừ ra ngay tại `e`). Đáp án là max của tổng tiền tố.
#### Code demo

In [12]:
def rooms_difference_array(intervals):
    m = max(e for _, e in intervals)
    diff = [0] * (m + 1)
    for s, e in intervals:
        diff[s] += 1
        diff[e] -= 1                                  # nửa mở: tại e cuộc họp không còn
    current = best = 0
    for value in diff:
        current += value
        best = max(best, current)
    return best


for intervals, expected in examples:
    print(intervals, "->", rooms_difference_array(intervals), "(kỳ vọng", expected, ")")

[[0, 30], [5, 10], [15, 20]] -> 2 (kỳ vọng 2 )
[[7, 10], [2, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n + M)`** với `M = max end <= 10^6` (không cần sắp xếp), **Space `O(M)`**. Nhanh khi `n` lớn và miền thời gian nhỏ; không dùng được khi thời gian có giá trị rất lớn hoặc số thực (khi đó dùng Cách 2/3/4).

> Cách 2 (heap) là bản nộp kinh điển. Cách 3 gọn và không cần heap. Cách 5 là mẹo mạnh cho miền nguyên nhỏ.

## 8. TRACE từng bước

Ta trace Cách 2 (heap) trên ví dụ 1 và trên một ca hòa `[[1,5],[5,8],[2,6]]`; rồi trace Cách 3 (hai con trỏ) trên ví dụ 1.
#### Code demo

In [13]:
def trace_heap(intervals):
    print("intervals =", intervals)
    ordered = sorted(intervals, key=lambda x: x[0])
    print("sau khi sắp theo giờ bắt đầu:", ordered)
    heap = []
    for s, e in ordered:
        before = sorted(heap)
        if heap and heap[0] <= s:
            old = heapq.heapreplace(heap, e)
            action = f"phòng rảnh (kết thúc {old} <= {s}) -> dùng lại, giờ kết thúc mới {e}"
        else:
            heapq.heappush(heap, e)
            action = f"mọi phòng bận (nhỏ nhất {before[0] if before else '-'} > {s}) -> mở phòng mới"
        print(f"  [{s},{e}]: heap trước={before} -> {action}; heap sau={sorted(heap)}")
    print("số phòng =", len(heap))
    return len(heap)


trace_heap([[0, 30], [5, 10], [15, 20]])
print()
trace_heap([[1, 5], [5, 8], [2, 6]])

intervals = [[0, 30], [5, 10], [15, 20]]
sau khi sắp theo giờ bắt đầu: [[0, 30], [5, 10], [15, 20]]
  [0,30]: heap trước=[] -> mọi phòng bận (nhỏ nhất - > 0) -> mở phòng mới; heap sau=[30]
  [5,10]: heap trước=[30] -> mọi phòng bận (nhỏ nhất 30 > 5) -> mở phòng mới; heap sau=[10, 30]
  [15,20]: heap trước=[10, 30] -> phòng rảnh (kết thúc 10 <= 15) -> dùng lại, giờ kết thúc mới 20; heap sau=[20, 30]
số phòng = 2

intervals = [[1, 5], [5, 8], [2, 6]]
sau khi sắp theo giờ bắt đầu: [[1, 5], [2, 6], [5, 8]]
  [1,5]: heap trước=[] -> mọi phòng bận (nhỏ nhất - > 1) -> mở phòng mới; heap sau=[5]
  [2,6]: heap trước=[5] -> mọi phòng bận (nhỏ nhất 5 > 2) -> mở phòng mới; heap sau=[5, 6]
  [5,8]: heap trước=[5, 6] -> phòng rảnh (kết thúc 5 <= 5) -> dùng lại, giờ kết thúc mới 8; heap sau=[6, 8]
số phòng = 2


2

In [14]:
def trace_two_pointers(intervals):
    starts = sorted(s for s, _ in intervals)
    ends = sorted(e for _, e in intervals)
    print("starts =", starts, "| ends =", ends)
    rooms, j = 0, 0
    for s in starts:
        if s < ends[j]:
            rooms += 1
            note = f"{s} < ends[{j}]={ends[j]}: chưa phòng nào rảnh -> rooms={rooms}"
        else:
            note = f"{s} >= ends[{j}]={ends[j]}: dùng lại phòng, j={j + 1}"
            j += 1
        print("  start", s, "->", note)
    print("số phòng =", rooms)
    return rooms


trace_two_pointers([[0, 30], [5, 10], [15, 20]])

starts = [0, 5, 15] | ends = [10, 20, 30]
  start 0 -> 0 < ends[0]=10: chưa phòng nào rảnh -> rooms=1
  start 5 -> 5 < ends[0]=10: chưa phòng nào rảnh -> rooms=2
  start 15 -> 15 >= ends[0]=10: dùng lại phòng, j=1
số phòng = 2


2

Bảng tóm tắt Cách 2 (ví dụ 1):

| cuộc họp (đã sắp) | heap trước | `heap[0] <= s`? | hành động | heap sau |
|---|---|---|---|---|
| `[0,30]` | `[]` | — | mở phòng | `[30]` |
| `[5,10]` | `[30]` | `30 <= 5` sai | mở phòng | `[10,30]` |
| `[15,20]` | `[10,30]` | `10 <= 15` đúng | dùng lại phòng, kết thúc mới `20` | `[20,30]` |

Kết quả `len(heap) = 2`. Ca hòa `[1,5]` rồi `[5,8]`: `heap[0] = 5 <= 5` đúng ⇒ dùng lại phòng (đây là lý do phải dùng `<=`).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (sắp xếp + min-heap). Đúng `class Solution`, đúng `minMeetingRooms(self, intervals: List[List[int]]) -> int`.

In [15]:
class Solution:
    def minMeetingRooms(self, intervals: List[List[int]]) -> int:
        heap = []                                          # giờ kết thúc của các phòng đang dùng
        for s, e in sorted(intervals, key=lambda x: x[0]):
            if heap and heap[0] <= s:                      # có phòng đã rảnh: dùng lại
                heapq.heapreplace(heap, e)
            else:                                          # không có: mở phòng mới
                heapq.heappush(heap, e)
        return len(heap)


sol = Solution()
for intervals, expected in examples:
    assert sol.minMeetingRooms(intervals) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Cận dưới:** nếu tại thời điểm `t` có `k` cuộc họp đang diễn ra thì cần `>= k` phòng; do đó đáp án `>=` độ sâu chồng lấn lớn nhất `D`.
- **Cận trên (thuật toán đạt `D`):** xét các cuộc họp theo giờ bắt đầu tăng dần. Heap chứa giờ kết thúc của đúng các phòng đang mở. Ngay trước khi xét `[s, e]`, **số phòng bận tại `s`** = số phần tử heap có giá trị `> s` (các phòng còn lại đã rảnh). Nếu `heap[0] <= s` thì có phòng rảnh, ta dùng lại (số phòng không tăng); nếu `heap[0] > s` thì **mọi** phòng đang mở đều bận tại `s` (vì `heap[0]` là nhỏ nhất), nên cuộc họp mới cộng thêm vào độ sâu tại `s` — buộc phải mở phòng mới, và lúc đó số phòng bằng độ sâu tại `s` (tối đa `D`).
- **Dùng phòng "rảnh sớm nhất" là đủ:** nếu có bất kỳ phòng nào rảnh thì phòng có giờ kết thúc nhỏ nhất chắc chắn rảnh; chọn phòng nào cũng cho số phòng tương lai như nhau (hoán đổi được).
- **`<=` (không phải `<`):** khoảng nửa mở nên `end == start` không chồng lấn.
- **Cách 3 (hai con trỏ):** `ends[j]` đóng vai `heap[0]` (giờ kết thúc nhỏ nhất chưa tái sử dụng); số lần `start < ends[j]` là số phòng phải mở.
- **Cách 4, 5:** độ sâu chồng lấn tính trực tiếp bằng tổng cộng dồn `+1/-1` (với `-1` trước `+1` khi hòa).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(intervals)`, `M = max end` (tối đa `10^6`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tại mỗi điểm bắt đầu | `O(n²)` | `O(1)` |
| 2. Sắp xếp + min-heap | `O(n log n)` | `O(n)` |
| 3. Hai mảng sắp, hai con trỏ | `O(n log n)` | `O(n)` |
| 4. Sự kiện quét | `O(n log n)` | `O(n)` |
| 5. Mảng hiệu | `O(n + M)` | `O(M)` |

Với `n = 10^4`: Cách 2–4 chỉ cỡ `10^5` phép; Cách 5 cỡ `10^6`; Cách 1 cỡ `10^8` (chậm). Không có khác biệt average/worst lớn (độ phức tạp do sắp xếp quyết định).

## 12. Bộ test

`oracle` là brute force **độc lập**: duyệt **mọi mốc thời gian nguyên** `t` từ `0` tới `max end` và đếm số cuộc họp có `start <= t < end`; đáp án là giá trị lớn nhất. Với tọa độ nhỏ nó rất rõ ràng (khác Cách 1 là chỉ xét điểm bắt đầu).
#### Code demo

In [16]:
def oracle(intervals):
    """Brute force độc lập: đếm tại MỌI mốc thời gian nguyên (miền nhỏ)."""
    last = max(e for _, e in intervals)
    best = 0
    for t in range(0, last + 1):
        running = sum(1 for s, e in intervals if s <= t < e)
        best = max(best, running)
    return best


impls = {
    "bruteforce": rooms_bruteforce,
    "heap": rooms_heap,
    "two_pointers": rooms_two_pointers,
    "sweep": rooms_sweep,
    "difference": rooms_difference_array,
    "submit": sol.minMeetingRooms,
}

for intervals, expected in examples:
    assert oracle(intervals) == expected
    for key, fn in impls.items():
        assert fn([list(x) for x in intervals]) == expected, (key, intervals, fn(intervals), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [17]:
edge_cases = [
    ([[1, 2]], 1, "một cuộc họp"),
    ([[1, 5], [5, 8]], 1, "kết thúc đúng lúc cuộc kia bắt đầu: không chồng"),
    ([[1, 5], [4, 8]], 2, "chồng lấn một phần"),
    ([[1, 10], [2, 3], [4, 5], [6, 7]], 2, "một cuộc họp dài chứa nhiều cuộc ngắn rời nhau"),
    ([[1, 4], [1, 4], [1, 4]], 3, "ba cuộc họp trùng hệt nhau"),
    ([[1, 2], [2, 3], [3, 4], [4, 5]], 1, "chuỗi liên tiếp nối đuôi"),
    ([[0, 1000000], [0, 1000000]], 2, "giá trị biên của đề"),
    ([[2, 7], [1, 3], [5, 9], [8, 10]], 2, "đầu vào chưa sắp xếp"),
    ([[1, 3], [2, 4], [3, 5], [4, 6]], 2, "mỗi thời điểm có tối đa 2 cuộc họp"),
    ([[5, 6], [1, 2], [3, 4]], 1, "rời nhau, đảo thứ tự"),
]
for intervals, expected, label in edge_cases:
    assert oracle(intervals) == expected, (label, oracle(intervals), expected)
    for key, fn in impls.items():
        assert fn([list(x) for x in intervals]) == expected, (key, label, fn(intervals), expected)
    print("PASS", label)

PASS một cuộc họp
PASS kết thúc đúng lúc cuộc kia bắt đầu: không chồng
PASS chồng lấn một phần
PASS một cuộc họp dài chứa nhiều cuộc ngắn rời nhau
PASS ba cuộc họp trùng hệt nhau
PASS chuỗi liên tiếp nối đuôi


PASS giá trị biên của đề
PASS đầu vào chưa sắp xếp
PASS mỗi thời điểm có tối đa 2 cuộc họp
PASS rời nhau, đảo thứ tự


### 12.2. Tính chất và phản ví dụ

- `1 <= đáp án <= n`.
- **Bất biến khi tịnh tiến thời gian** (cộng cùng một hằng vào mọi `start`, `end`) và khi **hoán vị thứ tự** các cuộc họp.
- **Đối xứng thời gian:** ánh xạ `[s, e) → [M - e, M - s)` không đổi đáp án.
- Nếu mọi cuộc họp **chung một điểm** thì đáp án `= n`; nếu **đôi một rời nhau** thì đáp án `= 1`.
- **Phản ví dụ** cho lỗi dùng `<` thay `<=`: `[[1,5],[5,8]]` có đáp án `1` nhưng bản sai trả `2`.
#### Code demo

In [18]:
random.seed(253)
for _ in range(500):
    n = random.randint(1, 12)
    intervals = []
    for _ in range(n):
        s = random.randint(0, 20)
        intervals.append([s, s + random.randint(1, 8)])
    ans = sol.minMeetingRooms(intervals)
    assert 1 <= ans <= n
    shift = random.randint(0, 50)
    assert sol.minMeetingRooms([[s + shift, e + shift] for s, e in intervals]) == ans
    shuffled = intervals[:]
    random.shuffle(shuffled)
    assert sol.minMeetingRooms(shuffled) == ans
    M = max(e for _, e in intervals)
    assert sol.minMeetingRooms([[M - e, M - s] for s, e in intervals]) == ans
print("PASS tính chất (cận, tịnh tiến, hoán vị, đối xứng thời gian) trên 500 bộ ngẫu nhiên")

common = [[random.randint(0, 4), random.randint(5, 9)] for _ in range(8)]       # đều chứa thời điểm 4.5 -> mọi họp chung điểm
assert sol.minMeetingRooms(common) == 8
disjoint = [[2 * i, 2 * i + 1] for i in range(8)]
assert sol.minMeetingRooms(disjoint) == 1
print("PASS chung một điểm -> n; đôi một rời nhau -> 1")


def sai_dau_nho_hon(intervals):
    heap = []
    for s, e in sorted(intervals, key=lambda x: x[0]):
        if heap and heap[0] < s:                          # SAI: phải là <=
            heapq.heapreplace(heap, e)
        else:
            heapq.heappush(heap, e)
    return len(heap)


print("phản ví dụ [[1,5],[5,8]]: đúng", sol.minMeetingRooms([[1, 5], [5, 8]]), "| dùng '<':", sai_dau_nho_hon([[1, 5], [5, 8]]))
assert sai_dau_nho_hon([[1, 5], [5, 8]]) == 2

PASS tính chất (cận, tịnh tiến, hoán vị, đối xứng thời gian) trên 500 bộ ngẫu nhiên
PASS chung một điểm -> n; đôi một rời nhau -> 1
phản ví dụ [[1,5],[5,8]]: đúng 1 | dùng '<': 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [19]:
random.seed(4253)
for trial in range(2500):
    n = random.randint(1, 15)
    span = random.choice([5, 10, 30])
    intervals = []
    for _ in range(n):
        s = random.randint(0, span)
        intervals.append([s, s + random.randint(1, span)])
    truth = oracle(intervals)
    for key, fn in impls.items():
        assert fn([list(x) for x in intervals]) == truth, (key, intervals, fn(intervals), truth)
print("PASS 2500 bộ ngẫu nhiên nhỏ: mọi cách khớp oracle (đếm tại mọi mốc nguyên)")

# Dữ liệu cỡ đề: n = 10^4, thời gian tới 10^6. Bỏ vét cạn O(n^2) trong Python thuần; so các cách còn lại.
random.seed(10_000)
big = []
for _ in range(10_000):
    s = random.randint(0, 999_000)
    big.append([s, random.randint(s + 1, min(s + random.choice([10, 1000, 100_000]), 1_000_000))])
ref = None
for key in ("heap", "two_pointers", "sweep", "difference", "submit"):
    t0 = time.perf_counter()
    value = impls[key]([list(x) for x in big])
    dt = (time.perf_counter() - t0) * 1000
    ref = value if ref is None else ref
    assert value == ref, (key, value, ref)
    print(f"{key:<13} ans={value:>5}  {dt:7.1f} ms")
print("PASS n = 10^4: các cách khớp nhau")

PASS 2500 bộ ngẫu nhiên nhỏ: mọi cách khớp oracle (đếm tại mọi mốc nguyên)
heap          ans=  202      4.4 ms
two_pointers  ans=  202      4.7 ms
sweep         ans=  202      6.9 ms
difference    ans=  202     35.8 ms
submit        ans=  202      4.2 ms
PASS n = 10^4: các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `<` thay vì `<=` khi so giờ kết thúc nhỏ nhất với giờ bắt đầu
Xem `sai_dau_nho_hon` ở phần test: `[[1,5],[5,8]]` trả `2` (sai). **Vì sao sai:** cuộc họp kết thúc lúc `5` và cuộc họp khác bắt đầu lúc `5` **không** chồng lấn. **Code đúng:** `heap[0] <= s`.

### Lỗi 2: Quên sắp xếp theo giờ bắt đầu
#### Code sai

In [20]:
def sai_khong_sap_xep(intervals):
    heap = []
    for s, e in intervals:                                # SAI: không sắp xếp theo giờ bắt đầu
        if heap and heap[0] <= s:
            heapq.heapreplace(heap, e)
        else:
            heapq.heappush(heap, e)
    return len(heap)


bad = [[15, 20], [0, 30], [5, 10]]
print("đúng:", sol.minMeetingRooms(bad), "| không sắp xếp:", sai_khong_sap_xep(bad))
bad2 = [[7, 10], [2, 4]]
print("đúng:", sol.minMeetingRooms(bad2), "| không sắp xếp:", sai_khong_sap_xep(bad2))

đúng: 2 | không sắp xếp: 3
đúng: 1 | không sắp xếp: 2


**Vì sao sai:** thuật toán giả định xử lý cuộc họp theo **thứ tự thời gian**; nếu không sắp xếp, "phòng rảnh sớm nhất" có thể ứng với cuộc họp ở **tương lai** so với cuộc họp đang xét. **Code đúng:** `sorted(intervals, key=lambda x: x[0])`.

### Lỗi 3: Đếm số cặp chồng lấn thay vì độ sâu chồng lấn
#### Code sai

In [21]:
def sai_dem_cap(intervals):
    count = 0
    for i in range(len(intervals)):
        for j in range(i + 1, len(intervals)):
            if overlap(intervals[i], intervals[j]):
                count += 1
    return max(1, count)                                  # SAI: số cặp chồng lấn không phải số phòng


bad = [[0, 10], [1, 2], [3, 4], [5, 6]]
print("đúng:", sol.minMeetingRooms(bad), "| số cặp chồng lấn:", sai_dem_cap(bad))

đúng: 2 | số cặp chồng lấn: 3


**Vì sao sai:** với `[[0,10],[1,2],[3,4],[5,6]]` có `3` cặp chồng lấn (cuộc họp dài với từng cuộc ngắn) nhưng chỉ cần `2` phòng. Đáp án là **số cuộc họp đồng thời tại một thời điểm**, không phải số cặp chồng lấn.

### Lỗi 4: Sự kiện hòa xếp kết thúc **sau** bắt đầu
#### Code sai

In [22]:
def sai_su_kien_hoa(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e, -1))
    events.sort(key=lambda x: (x[0], -x[1]))              # SAI: khi hòa, đặt +1 (bắt đầu) TRƯỚC -1 (kết thúc)
    current = best = 0
    for _, delta in events:
        current += delta
        best = max(best, current)
    return best


print("[[1,5],[5,8]] đúng:", sol.minMeetingRooms([[1, 5], [5, 8]]), "| sự kiện hòa sai thứ tự:", sai_su_kien_hoa([[1, 5], [5, 8]]))

[[1,5],[5,8]] đúng: 1 | sự kiện hòa sai thứ tự: 2


**Vì sao sai:** khi sự kiện bắt đầu tại `5` đứng trước sự kiện kết thúc tại `5`, tổng cộng dồn tạm tăng lên `2` trước khi giảm — đếm nhầm một chồng lấn không có thật. **Code đúng:** mã `-1` cho kết thúc, `+1` cho bắt đầu và sắp bộ `(t, delta)` mặc định — khi hòa `-1 < +1` nên kết thúc đứng **trước**.

### Lỗi 5: Mảng hiệu coi giờ kết thúc là "bao gồm" (`diff[e + 1] -= 1`)
#### Code sai

In [23]:
def sai_mang_hieu(intervals):
    m = max(e for _, e in intervals)
    diff = [0] * (m + 2)
    for s, e in intervals:
        diff[s] += 1
        diff[e + 1] -= 1                                      # SAI: khoảng là nửa mở, phải trừ tại e
    current = best = 0
    for value in diff:
        current += value
        best = max(best, current)
    return best


print("đúng:", sol.minMeetingRooms([[1, 5], [5, 8]]), "| sai:", sai_mang_hieu([[1, 5], [5, 8]]))
print("đúng:", sol.minMeetingRooms([[0, 30], [5, 10], [15, 20]]), "| sai:", sai_mang_hieu([[0, 30], [5, 10], [15, 20]]))

đúng: 1 | sai: 2
đúng: 2 | sai: 2


**Vì sao sai:** nếu trừ tại `e + 1` thì cuộc họp vẫn "đang diễn ra" tại thời điểm `e`, nên `[1,5]` và `[5,8]` bị coi là chồng nhau tại `5`. Với khoảng nửa mở `[s, e)`: `diff[s] += 1`, `diff[e] -= 1` và kích thước mảng `max_end + 1`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**số tài nguyên tối thiểu để phục vụ các khoảng thời gian chồng nhau**" (phòng họp, nhân viên, CPU, sân bay) → **đáp án là độ sâu chồng lấn lớn nhất**: dùng **heap**, **hai con trỏ trên hai mảng sắp** hoặc **sự kiện quét**.
- Nhớ quy ước **nửa mở**: `end == start` không chồng lấn → dùng `<=` khi so (heap) / `-1` trước `+1` (sự kiện).
- Nếu miền thời gian là **số nguyên nhỏ** → **mảng hiệu**; thời gian lớn/thực → sắp xếp.
- Họ hàng: **Meeting Rooms I** (252: có chồng lấn không), **Merge Intervals** (56), **Non-overlapping Intervals** (435), **Car Pooling** (1094), **My Calendar** (729–731), **Minimum Number of Arrows** (452).
- Luôn **sắp xếp trước** (theo giờ bắt đầu) trừ khi dùng mảng hiệu.
- Tự thử: các cuộc họp chung một điểm (`= n`), rời nhau (`= 1`), kết thúc đúng lúc bắt đầu.

## 15. Biến thể của bài

1. **Meeting Rooms** (LeetCode 252): có thể tham dự mọi cuộc họp không (không chồng lấn).
2. **Car Pooling** (1094): mỗi khoảng có trọng số (số hành khách) — mảng hiệu / sự kiện với `delta` khác `±1`.
3. **My Calendar III** (732): độ sâu chồng lấn lớn nhất theo thời gian thực — cây đoạn / map sự kiện.
4. **Gán phòng cụ thể**: trả về cuộc họp nào ở phòng nào (heap lưu cả chỉ số phòng).
5. **Thời điểm** có nhiều cuộc họp đồng thời nhất.
#### Code demo

In [24]:
def can_attend_all(intervals):
    """LeetCode 252: không có hai cuộc họp nào chồng lấn."""
    ordered = sorted(intervals)
    return all(ordered[i][1] <= ordered[i + 1][0] for i in range(len(ordered) - 1))


def assign_rooms(intervals):
    """Trả (số phòng, danh sách phòng của từng cuộc họp theo thứ tự đầu vào) bằng heap (giờ kết thúc, số phòng)."""
    order = sorted(range(len(intervals)), key=lambda i: intervals[i][0])
    heap, free, room_of, next_room = [], [], [0] * len(intervals), 0
    for idx in order:
        s, e = intervals[idx]
        if heap and heap[0][0] <= s:
            _, room = heapq.heapreplace(heap, (e, heap[0][1]))
        else:
            room = next_room
            next_room += 1
            heapq.heappush(heap, (e, room))
        room_of[idx] = room
    return next_room, room_of


def busiest_time(intervals):
    """Thời điểm (nhỏ nhất) có nhiều cuộc họp nhất và số cuộc họp đó."""
    events = sorted([(s, 1) for s, _ in intervals] + [(e, -1) for _, e in intervals])
    cur = best = 0
    when = None
    for t, d in events:
        cur += d
        if cur > best:
            best, when = cur, t
    return when, best


print(can_attend_all([[7, 10], [2, 4]]), can_attend_all([[0, 30], [5, 10], [15, 20]]))     # True False
print(assign_rooms([[0, 30], [5, 10], [15, 20]]))                                          # (2, [0, 1, 1])
print(busiest_time([[0, 30], [5, 10], [15, 20]]))                                          # (5, 2)

True False
(2, [0, 1, 1])
(5, 2)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | đếm chồng lấn tại mỗi điểm bắt đầu | `O(n²)` | `O(1)` | oracle, `n` nhỏ |
| 2. Sắp xếp + min-heap | heap giờ kết thúc, dùng lại phòng rảnh | `O(n log n)` | `O(n)` | **cách nộp chuẩn**, cần biết phòng nào |
| 3. Hai con trỏ trên hai mảng sắp | `start < ends[j]` thì mở phòng | `O(n log n)` | `O(n)` | gọn, không heap |
| 4. Sự kiện quét | `(t, ±1)` sắp xếp, cộng dồn | `O(n log n)` | `O(n)` | khái quát cho trọng số |
| 5. Mảng hiệu | `diff[s]++`, `diff[e]--` theo miền nguyên | `O(n + M)` | `O(M)` | miền thời gian nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `overlap([1, 5], [5, 9])`; (b) `sorted([(5, 1), (5, -1), (3, 1)])`; (c) `Solution().minMeetingRooms([[1, 2], [2, 3]])`; (d) `Solution().minMeetingRooms([[1, 3], [2, 4], [3, 5]])`.

**Bài 2 (tính bằng tay).** Với `[[1, 4], [2, 3], [3, 6], [5, 7]]`: vẽ trục thời gian, liệt kê số cuộc họp đang diễn ra tại `t = 1, 2, 3, 4, 5, 6` và nêu đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `rooms_todo(intervals)` theo Cách 2 (min-heap).
#### Code demo

In [25]:
def rooms_todo(intervals):
    heap = []
    # TODO 1: duyệt (s, e) trong intervals đã sắp theo giờ bắt đầu
    # TODO 2: nếu heap không rỗng và heap[0] <= s -> heapq.heapreplace(heap, e)
    # TODO 3: ngược lại -> heapq.heappush(heap, e)
    # TODO 4: trả len(heap)
    pass


# Tự làm xong rồi mới mở test:
# for intervals, expected in examples:
#     assert rooms_todo(intervals) == expected

**Bài 4 (tự viết lại).** Viết `rooms_sweep2(intervals)` bằng sự kiện quét, **không nhìn lại** Cách 4; kiểm tra bằng `oracle` trên dữ liệu ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** Cho thêm số nguyên `k` (số phòng hiện có). Viết `can_schedule(intervals, k)` trả `True` nếu `k` phòng là đủ. Sau đó viết `max_meetings_with_k_rooms(intervals, k)` chọn **nhiều cuộc họp nhất** có thể xếp vào `k` phòng (gợi ý greedy: sắp theo giờ kết thúc, gán vào phòng rảnh có giờ kết thúc lớn nhất nhưng `<= s`); thử `k = 1` trên `[[0,30],[5,10],[15,20]]` → `2`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: chồng lấn cần `s1 < e2` và `s2 < e1` (nửa mở); sắp bộ so thành phần đầu rồi thứ hai. Bài 2: đếm cuộc họp có `start <= t < end`. Bài 5: `can_schedule` chỉ là `minMeetingRooms <= k`.
- **Mức 2 (vừa):** Bài 1: `False`, `[(3, 1), (5, -1), (5, 1)]`, `1`, `2`. Bài 2: `t=1`:1, `t=2`:2, `t=3`:2 (`[1,4]`, `[3,6]`; `[2,3]` đã kết thúc), `t=4`:1, `t=5`:2 (`[3,6]`, `[5,7]`), `t=6`:1 ⇒ đáp án `2`. Bài 3: khung Cách 2.
- **Mức 3 (sâu):** Bài 4: `events.sort()` rồi `current += d`; nhớ `-1` cho kết thúc. Bài 5: greedy theo giờ kết thúc: duy trì danh sách giờ kết thúc của `k` phòng (có thể bắt đầu `-inf`), với cuộc họp `[s, e]` chọn phòng có giờ kết thúc lớn nhất `<= s` (dùng `bisect`); nếu không có thì bỏ cuộc họp.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
import bisect

# Bài 1
assert overlap([1, 5], [5, 9]) is False
assert sorted([(5, 1), (5, -1), (3, 1)]) == [(3, 1), (5, -1), (5, 1)]
assert Solution().minMeetingRooms([[1, 2], [2, 3]]) == 1
assert Solution().minMeetingRooms([[1, 3], [2, 4], [3, 5]]) == 2

# Bài 2
b2 = [[1, 4], [2, 3], [3, 6], [5, 7]]
counts = [sum(1 for s, e in b2 if s <= t < e) for t in range(1, 7)]
assert counts == [1, 2, 2, 1, 2, 1] and sol.minMeetingRooms(b2) == 2 == oracle(b2)


# Bài 3
def rooms_todo_ans(intervals):
    heap = []
    for s, e in sorted(intervals, key=lambda x: x[0]):
        if heap and heap[0] <= s:
            heapq.heapreplace(heap, e)
        else:
            heapq.heappush(heap, e)
    return len(heap)


for intervals, expected in examples:
    assert rooms_todo_ans(intervals) == expected


# Bài 4
def rooms_sweep2(intervals):
    events = sorted([(s, 1) for s, _ in intervals] + [(e, -1) for _, e in intervals])
    cur = best = 0
    for _, d in events:
        cur += d
        best = max(best, cur)
    return best


random.seed(10)
for _ in range(300):
    n = random.randint(1, 10)
    iv = [[s, s + random.randint(1, 6)] for s in (random.randint(0, 15) for _ in range(n))]
    assert rooms_sweep2(iv) == oracle(iv)


# Bài 5
def can_schedule(intervals, k):
    return Solution().minMeetingRooms(intervals) <= k


def max_meetings_with_k_rooms(intervals, k):
    ends = [float("-inf")] * k                          # giờ kết thúc của từng phòng, luôn giữ tăng dần
    taken = 0
    for s, e in sorted(intervals, key=lambda x: x[1]):
        idx = bisect.bisect_right(ends, s) - 1          # phòng có giờ kết thúc lớn nhất <= s
        if idx >= 0:
            ends[idx] = e
            ends.sort()
            taken += 1
    return taken


assert can_schedule([[0, 30], [5, 10], [15, 20]], 2) and not can_schedule([[0, 30], [5, 10], [15, 20]], 1)
assert max_meetings_with_k_rooms([[0, 30], [5, 10], [15, 20]], 1) == 2
assert max_meetings_with_k_rooms([[0, 30], [5, 10], [15, 20]], 2) == 3
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp + min-heap giờ kết thúc)

```python
import heapq

class Solution:
    def minMeetingRooms(self, intervals: List[List[int]]) -> int:
        heap = []
        for s, e in sorted(intervals, key=lambda x: x[0]):
            if heap and heap[0] <= s:
                heapq.heapreplace(heap, e)
            else:
                heapq.heappush(heap, e)
        return len(heap)
```

**Checklist**

- [ ] Hiểu đáp án = **số cuộc họp đồng thời nhiều nhất**?
- [ ] Quy ước **nửa mở**: `end == start` không chồng lấn → `heap[0] <= s` (hoặc `-1` trước `+1`)?
- [ ] Đã **sắp xếp theo giờ bắt đầu** trước khi dùng heap?
- [ ] Heap chứa **giờ kết thúc**, phòng rảnh sớm nhất là `heap[0]`?
- [ ] Đã thử một cuộc họp, họp trùng hệt, nối đuôi nhau, một họp dài chứa nhiều họp ngắn?
- [ ] Đã đối chiếu với brute force đếm tại mọi mốc thời gian?
- [ ] Skeleton đúng `class Solution` / `def minMeetingRooms(self, intervals: List[List[int]]) -> int`?